# NER LLM Finetuned
This pipeline uses an LLM trained on NER input-output pairs, using a chat template

**WIP**

In [ ]:
import pandas as pd

data = pd.read_json(path_or_buf="../data/labelled/all.jsonl", lines=True)
data = data.rename({"text": "input"}, axis=1)

data

In [ ]:
sentences = list(data["input"])
tags = list(data["label"])

for t in tags:
    for l in range(len(t), 0, -1):
        start, end, label = t[l - 1]

        if label.startswith("I"):
            t[l - 2][1] = t[l - 1][1]


clean_tags = []
for t in tags:
    clean_tags.append([tag for tag in t if not tag[2].startswith("I")])

In [ ]:
# le persone sono etichettate tra doppie parentesi tonde: ((PER))
# i luoghi sono etichettati tra doppie parentesi quadre: [[LOC]]
# le organizzazioni sono etichettate tra doppie parentesi graffe: {{ORG}}
# le misc sono etichettate tra doppie parentesi angolari: <<MISC>>
brackets = {
    "B-PER": ["((", "))"],
    "B-LOC": ["[[", "]]"],
    "B-ORG": ["{{", "}}"],
    "B-MISC": ["<<", ">>"],
    "I-PER": ["", ""],
    "I-LOC": ["", ""],
    "I-ORG": ["", ""],
    "I-MISC": ["", ""],
}

# Add brackets between NER tokens
ner_sentences = []
for s, tags in zip(sentences, clean_tags):
    new = s
    og_len = len(new)
    for i, t in enumerate(tags):
        start_idx, end_idx, label = t[0], t[1], t[2]
        word = s[start_idx:end_idx]
        new = (
            new[: start_idx + len(new) - og_len]
            + brackets[label][0]
            + word
            + brackets[label][1]
            + new[len(word) + start_idx + len(new) - og_len :]
        )

    ner_sentences.append(new)

# Remove unnecessary newlines
ner_sentences = [s.replace("\n", "") for s in ner_sentences]


In [ ]:
data["text"] = ner_sentences

data.drop(
    ["meta", "label"],
    axis=1,
).to_json(
    "../data/labelled/all_parallel.jsonl",
    orient="records",
    lines=True,
)

data